# Flat evaluation reporting demo

Demonstrates **Layer S** (structural row pairing), **Layer 1** (applicability), and **Layer 2** (value matching) on a collated checklist document.

**Model schema** (`schema.json`) — what one model call returns:
- Root key `outputs[]`: panel rows (`label`, `is_micrograph`, `caption_snippet`)

**Evaluation gold/pred** — collation wrappers around repeated model calls:
- `papers[]` → `figures[]` (structural, positional join)
- each figure embeds `outputs[]` (predictive, Hungarian alignment on `label`)

The notebook uses `FlatEvaluator` (phase 5 + 5.1) rather than wiring modules by hand.

Toy errors in pred:
- Figure 1 — `1A.is_micrograph` false positive (gold `no`, pred `yes`)
- Figure 2 — reordered outputs; wrong `2A.is_micrograph`; `2A.caption_snippet` mismatch
- Figure 3 — 3B missing; spurious 3C

In [ ]:
from __future__ import annotations

import json
from pathlib import Path

import pandas as pd
import plotly.express as px
import plotly.graph_objects as go
from plotly.subplots import make_subplots

from soda_mmqc.core.collation import discover_collation_layout
from soda_mmqc.core.eval_manifest import MatchingMetric
from soda_mmqc.core.evaluation import FlatEvaluator, format_ancestor_context


In [ ]:
def find_repo_root() -> Path:
    here = Path.cwd().resolve()
    for candidate in [here, *here.parents]:
        if (candidate / "soda_mmqc").is_dir() and (candidate / "pyproject.toml").is_file():
            return candidate
    return here


ROOT = find_repo_root()
DEMO_DIR = ROOT / "notebooks/fixtures/flat-eval-demo"

evaluator = FlatEvaluator.from_paths(
    str(DEMO_DIR / "schema.json"),
    str(DEMO_DIR / "manifest.json"),
)
gold = json.loads((DEMO_DIR / "gold.json").read_text())
pred = json.loads((DEMO_DIR / "pred.json").read_text())

layout = discover_collation_layout(evaluator.schema, gold, pred)
result = evaluator.evaluate(gold, pred)
manifest = evaluator.manifest
PREDICTIVE_LIST_KEY = layout.predictive_lists[0].by_list_key

print(f"Checklist: {manifest.checklist}")
print(f"Embedding prefix: {'.'.join(layout.embedding_prefix)}")
print(f"Predictive list: {PREDICTIVE_LIST_KEY}")
print(f"Profiled leaves: {manifest.profiled_leaf_properties()}")

## Fixture overview

| Figure | Gold | Pred (errors) |
|--------|------|---------------|
| 1 | Panel 1A not a micrograph | `1A.is_micrograph` FP (pred `yes`) |
| 2 | Outputs 2A/2B | Reordered; `2A.is_micrograph` FN; `2A.caption_snippet` mismatch |
| 3 | Outputs 3A/3B | 3A correct; 3B missing; spurious 3C |

In [ ]:
pd.set_option("display.max_colwidth", 60)

display(pd.DataFrame({"schema (model call)": [json.dumps(evaluator.schema, indent=2)]}))
display(pd.DataFrame({"manifest": [json.dumps(json.loads((DEMO_DIR / "manifest.json").read_text()), indent=2)]}))
display(pd.DataFrame({"gold": [json.dumps(gold, indent=2)], "pred": [json.dumps(pred, indent=2)]}))

In [ ]:
LAYER_S_ORDER = ("correct_row", "missing_row", "spurious_row")
LAYER1_ORDER = (
    "correct_NA",
    "correct_applicable",
    "withheld_applicable",
    "spurious_applicable",
)
LAYER2_BINARY_ORDER = ("TP", "TN", "FP", "FN")
LAYER2_GRADED_ORDER = ("match", "mismatch")

LAYER_S_COLORS = {
    "correct_row": "#2563eb",
    "missing_row": "#60a5fa",
    "spurious_row": "#1e3a8a",
}
LAYER1_COLORS = {
    "correct_NA": "#d6d3d1",
    "correct_applicable": "#f59e0b",
    "withheld_applicable": "#fcd34d",
    "spurious_applicable": "#b45309",
}
LAYER2_BINARY_COLORS = {
    "TP": "#16a34a",
    "TN": "#86efac",
    "FP": "#dc2626",
    "FN": "#ea580c",
}
LAYER2_GRADED_COLORS = {
    "match": "#7c3aed",
    "mismatch": "#f472b6",
}


def leaf_property_tail(leaf_property: str) -> str:
    return leaf_property.rsplit(".", maxsplit=1)[-1]


def counts_to_frame(counts, order: tuple[str, ...], label: str) -> pd.DataFrame:
    return pd.DataFrame({label: list(order), "count": [counts.get(key, 0) for key in order]})


def plot_outcome_bar(
    counts,
    order: tuple[str, ...],
    *,
    title: str,
    x_label: str,
    color_map: dict[str, str],
):
    labels = list(order)
    values = [counts.get(label, 0) for label in labels]
    colors = [color_map.get(label, "#7f7f7f") for label in labels]
    fig = go.Figure(go.Bar(x=labels, y=values, marker_color=colors))
    fig.update_layout(title=title, xaxis_title=x_label, yaxis_title="count", yaxis=dict(rangemode="tozero"))
    return fig


def layer_counts_by_property(result, order: tuple[str, ...], attr: str) -> pd.DataFrame:
    """Wide table of reporting counts per manifest leaf property."""
    rows = []
    for leaf_property, summary in sorted(result.by_property.items()):
        counts = getattr(summary, attr)
        if not counts:
            continue
        rows.append(
            {
                "leaf_property": leaf_property,
                "field": leaf_property_tail(leaf_property),
                **{key: counts.get(key, 0) for key in order},
            }
        )
    return pd.DataFrame(rows)


def split_layer2_by_metric(result, manifest) -> tuple[pd.DataFrame, pd.DataFrame]:
    """Split layer-2 counts into binary_polarity vs graded_string properties."""
    binary_rows = []
    graded_rows = []
    for leaf_property, summary in sorted(result.by_property.items()):
        if not summary.layer2_counts:
            continue
        profile = manifest.profile_for(leaf_property)
        if profile is None or not profile.is_profiled:
            continue
        base = {
            "leaf_property": leaf_property,
            "field": leaf_property_tail(leaf_property),
        }
        if profile.matching_metric == MatchingMetric.BINARY_POLARITY:
            binary_rows.append(
                {
                    **base,
                    **{key: summary.layer2_counts.get(key, 0) for key in LAYER2_BINARY_ORDER},
                }
            )
        elif profile.matching_metric == MatchingMetric.GRADED_STRING:
            graded_rows.append(
                {
                    **base,
                    **{key: summary.layer2_counts.get(key, 0) for key in LAYER2_GRADED_ORDER},
                }
            )
    return pd.DataFrame(binary_rows), pd.DataFrame(graded_rows)


def plot_layer2_stacked(
    df: pd.DataFrame,
    order: tuple[str, ...],
    *,
    title: str,
    color_map: dict[str, str],
) -> go.Figure:
    melted = df.melt(
        id_vars=["leaf_property", "field"],
        value_vars=list(order),
        var_name="layer2",
        value_name="count",
    )
    melted = melted[melted["count"] > 0]
    field_order = df["field"].tolist()
    fig = px.bar(
        melted,
        x="field",
        y="count",
        color="layer2",
        barmode="stack",
        category_orders={"layer2": list(order), "field": field_order},
        color_discrete_map=color_map,
        title=title,
        labels={"field": "leaf field", "layer2": "layer 2 outcome"},
    )
    fig.update_layout(yaxis=dict(rangemode="tozero"))
    return fig


instances_df = pd.DataFrame([instance.to_dict() for instance in result.instances]).rename(
    columns={"path": "instance_path", "exp_value": "gold", "pred_value": "pred"}
)
layer_s_payload = result.by_list[PREDICTIVE_LIST_KEY]
layer_s_counts = layer_s_payload["row_counts"]
layer1_by_property = layer_counts_by_property(result, LAYER1_ORDER, "layer1_counts")
layer2_binary_by_property, layer2_graded_by_property = split_layer2_by_metric(
    result, evaluator.manifest
)

print("Instance counts by leaf property:")
display(
    instances_df.groupby(["leaf_property", "layer1", "layer2"], dropna=False)
    .size()
    .reset_index(name="count")
)



## Layer S — predictive list row reporting (`by_list`)

Row-slot outcomes for `papers[].figures[].outputs[]` (Hungarian pairing on `label`).
Structural lists (`papers[]`, `figures[]`) use positional join and do not appear in `by_list`.

Missing and spurious rows include paper / figure / panel labels so you can see the culprit immediately.

In [ ]:
layer_s_summary = counts_to_frame(layer_s_counts, LAYER_S_ORDER, "structural")
display(layer_s_summary)

fig_s = plot_outcome_bar(
    layer_s_counts,
    LAYER_S_ORDER,
    title=f"Layer S — {PREDICTIVE_LIST_KEY} row outcomes (all figures)",
    x_label="structural outcome",
    color_map=LAYER_S_COLORS,
)
fig_s.show()

issues = result.layer_s_issues(PREDICTIVE_LIST_KEY)


def issue_table(issue_rows, *, side: str, alignment_col: str) -> pd.DataFrame:
    ancestor_col = f"ancestor_{side}"
    rows = []
    for row in issue_rows:
        rows.append(
            {
                "location": format_ancestor_context(row[ancestor_col]),
                alignment_col: row[f"{side}_alignment"],
                "context_path": row["context_path"],
            }
        )
    return pd.DataFrame(rows)


missing_df = issue_table(issues["missing"], side="gold", alignment_col="gold_row")
spurious_df = issue_table(issues["spurious"], side="pred", alignment_col="pred_row")

print("Missing rows (gold row with no pred match)")
display(missing_df if not missing_df.empty else pd.DataFrame(columns=["location", "gold_row", "context_path"]))

print("Spurious rows (pred row with no gold match)")
display(spurious_df if not spurious_df.empty else pd.DataFrame(columns=["location", "pred_row", "context_path"]))

## Layer 1 — applicability reporting

Was the field answered when it should (or should not) have been?

Counts are reported **per leaf property** (`by_property`) — never pooled across properties.

In [ ]:
display(layer1_by_property.set_index("leaf_property"))
display(
    instances_df.sort_values(["leaf_property", "instance_path"])[
        ["leaf_property", "instance_path", "gold", "pred", "layer1"]
    ]
)

layer1_melted = layer1_by_property.melt(
    id_vars=["leaf_property", "field"],
    value_vars=list(LAYER1_ORDER),
    var_name="layer1",
    value_name="count",
)
layer1_melted = layer1_melted[layer1_melted["count"] > 0]
field_order = layer1_by_property["field"].tolist()

fig_l1 = px.bar(
    layer1_melted,
    x="field",
    y="count",
    color="layer1",
    barmode="stack",
    category_orders={"layer1": list(LAYER1_ORDER), "field": field_order},
    color_discrete_map=LAYER1_COLORS,
    title="Layer 1 — applicability outcomes by leaf field",
    labels={"field": "leaf field", "layer1": "layer 1 outcome"},
)
fig_l1.update_layout(yaxis=dict(rangemode="tozero"))
fig_l1.show()

fig_l1_facet = px.bar(
    layer1_melted,
    x="layer1",
    y="count",
    color="layer1",
    facet_col="field",
    category_orders={"layer1": list(LAYER1_ORDER), "field": field_order},
    color_discrete_map=LAYER1_COLORS,
    title="Layer 1 — applicability outcomes (one panel per leaf field)",
)
fig_l1_facet.update_layout(showlegend=False, yaxis=dict(rangemode="tozero"))
fig_l1_facet.for_each_yaxis(lambda axis: axis.update(matches=None))
fig_l1_facet.show()

## Layer 2 — value matching reporting

Only instances with `layer1 = correct_applicable` receive a layer 2 label.

Layer 2 outcomes depend on the manifest `matching_metric`:
- **binary_polarity** → TP / TN / FP / FN
- **graded_string** → match / mismatch

Plots are split by metric family — never pooled across properties or outcome vocabularies.

In [ ]:
layer2_eligible = instances_df[instances_df["layer1"] == "correct_applicable"].copy()

print("binary_polarity properties")
display(layer2_binary_by_property.set_index("leaf_property"))
print("graded_string properties")
display(layer2_graded_by_property.set_index("leaf_property"))
display(
    layer2_eligible.sort_values(["leaf_property", "instance_path"])[
        ["leaf_property", "instance_path", "gold", "pred", "score", "layer2"]
    ]
)

if not layer2_binary_by_property.empty:
    plot_layer2_stacked(
        layer2_binary_by_property,
        LAYER2_BINARY_ORDER,
        title="Layer 2 — binary polarity (TP/TN/FP/FN) by leaf field",
        color_map=LAYER2_BINARY_COLORS,
    ).show()
else:
    print("No binary_polarity layer-2 counts")

if not layer2_graded_by_property.empty:
    plot_layer2_stacked(
        layer2_graded_by_property,
        LAYER2_GRADED_ORDER,
        title="Layer 2 — graded string (match/mismatch) by leaf field",
        color_map=LAYER2_GRADED_COLORS,
    ).show()
else:
    print("No graded_string layer-2 counts")

## Combined dashboard

Layer S, Layer 1, and Layer 2 (binary vs graded) stacked by leaf field.

In [ ]:
DASHBOARD_LEGEND_LAYOUT = {
    2: dict(orientation="h", yanchor="bottom", y=1.14, xanchor="center", x=0.36, font=dict(size=9)),
    3: dict(orientation="h", yanchor="bottom", y=1.14, xanchor="center", x=0.61, font=dict(size=9)),
    4: dict(orientation="h", yanchor="bottom", y=1.14, xanchor="center", x=0.86, font=dict(size=9)),
}

fig = make_subplots(
    rows=1,
    cols=4,
    subplot_titles=(
        "Layer S",
        "Layer 1",
        "Layer 2 binary",
        "Layer 2 graded",
    ),
)

layer_s_labels = list(LAYER_S_ORDER)
layer_s_values = [layer_s_counts.get(label, 0) for label in layer_s_labels]
layer_s_bar_colors = [LAYER_S_COLORS[label] for label in layer_s_labels]
fig.add_trace(
    go.Bar(x=layer_s_labels, y=layer_s_values, marker_color=layer_s_bar_colors, showlegend=False),
    row=1,
    col=1,
)

for col, by_property, order, color_map in (
    (2, layer1_by_property, LAYER1_ORDER, LAYER1_COLORS),
    (3, layer2_binary_by_property, LAYER2_BINARY_ORDER, LAYER2_BINARY_COLORS),
    (4, layer2_graded_by_property, LAYER2_GRADED_ORDER, LAYER2_GRADED_COLORS),
):
    if by_property.empty:
        continue
    fields = by_property["field"].tolist()
    stack_base = [0] * len(fields)
    legend_name = f"legend{col}"
    for outcome in order:
        values = by_property[outcome].tolist() if outcome in by_property else [0] * len(fields)
        if sum(values) == 0:
            continue
        fig.add_trace(
            go.Bar(
                x=fields,
                y=values,
                base=stack_base,
                name=outcome,
                marker_color=color_map[outcome],
                legend=legend_name,
                showlegend=True,
                legendgroup=legend_name,
            ),
            row=1,
            col=col,
        )
        stack_base = [base + value for base, value in zip(stack_base, values)]

fig.update_layout(
    title_text="Flat evaluation reporting — Layer S, Layer 1, Layer 2 (per leaf field)",
    height=460,
    showlegend=False,
    legend2=DASHBOARD_LEGEND_LAYOUT[2],
    legend3=DASHBOARD_LEGEND_LAYOUT[3],
    legend4=DASHBOARD_LEGEND_LAYOUT[4],
    yaxis=dict(rangemode="tozero"),
    yaxis2=dict(rangemode="tozero"),
    yaxis3=dict(rangemode="tozero"),
    yaxis4=dict(rangemode="tozero"),
)
fig.show()